# Data Cleaning

# Student Performance Data Cleaning and Preprocessing

## Project Overview

This notebook focuses on cleaning, preprocessing, and transforming the Student Performance Factors dataset in preparation for exploratory data analysis (EDA) and machine learning modeling.

The dataset contains academic, behavioral, demographic, and lifestyle-related variables used to predict student exam performance.

## Objectives of This Notebook

The primary goals of this preprocessing pipeline are:

- Load and inspect the raw dataset
- Analyze dataset structure and feature types
- Identify missing values and inconsistencies
- Examine feature distributions and categorical values
- Encode categorical variables into numerical form
- Handle missing values using imputation techniques
- Engineer additional predictive features
- Prepare a clean dataset suitable for machine learning models

## Importance of Data Cleaning

Raw datasets often contain missing values, categorical inconsistencies, and unstructured information that machine learning models cannot process directly. Proper preprocessing improves:

- Model accuracy
- Training stability
- Interpretability
- Generalization performance

The cleaned dataset generated in this notebook is later used for ensemble regression modeling and performance evaluation.

## Loading and Inspecting the Dataset

The first step in the preprocessing pipeline is loading the dataset into a Pandas DataFrame and performing an initial inspection.

### Purpose of This Step

This inspection helps:

- Verify that the dataset loaded correctly
- Understand the structure of the data
- Identify the number of rows and columns
- Preview feature names and sample records
- Begin identifying numerical and categorical variables

Understanding the raw dataset structure is important before performing any preprocessing or machine learning tasks.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("StudentPerformanceFactors.csv")

df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [5]:
df.shape

(6607, 20)

## Dataset Structure and Data Types

The dataset structure is analyzed using `df.info()` to examine:

- Column names
- Data types
- Non-null counts
- Memory usage

### Why This Step Matters

Machine learning models require numerical input data. Understanding data types allows us to identify:

- Numerical features
- Categorical features
- Columns containing missing values
- Features that may require encoding or transformation

This step also helps determine which preprocessing methods should be applied to different feature groups.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   str  
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   str  
 8   Internet_Access             6607 non-null   str  
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   str  
 11  Teacher_Quality             6529 non-null   str  
 12  School_Type                 6607 non-null   str  
 13  Peer_Influence              6607 non-null   str  
 14  Physical_Activity  

## Interpretation of Dataset Structure

The dataset contains a mixture of numerical and categorical features related to student academic performance, behavior, and demographics.

### Key Findings

- The dataset contains over 6,600 student records and 20 total features.
- Several columns were stored as string/object data types and required encoding before modeling.
- Most columns contained complete data with very few missing values.
- Missing values were identified primarily in:
  - `Parental_Education_Level`
  - `Teacher_Quality`
  - `Distance_from_Home`

### What This Means

Since machine learning models cannot directly process categorical string values, these variables needed to be converted into numerical representations. Additionally, missing values needed to be handled carefully to avoid information loss and maintain model stability.

## Missing Value Analysis

Before training machine learning models, it is important to identify incomplete or missing data within the dataset.

### Purpose of This Step

This analysis helps:

- Detect columns containing null values
- Measure the percentage of missing data
- Determine whether imputation is necessary
- Evaluate the overall quality of the dataset

Handling missing values properly is essential because many machine learning algorithms cannot process null entries directly.

In [9]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percent
})

missing_df.sort_values(by="Missing %", ascending=False)

,Missing Count,Missing %
Parental_Education_Level,90,1.362192
Teacher_Quality,78,1.180566
Distance_from_Home,67,1.014076
Hours_Studied,0,0.000000
Attendance,0,0.000000
Gender,0,0.000000
Learning_Disabilities,0,0.000000
Physical_Activity,0,0.000000
Peer_Influence,0,0.000000
School_Type,0,0.000000


## Interpretation of Missing Values

The missing value analysis revealed that only a small number of features contained incomplete observations.

### Key Findings

- Most columns contained no missing values.
- Missing values were concentrated in:
  - `Parental_Education_Level`
  - `Teacher_Quality`
  - `Distance_from_Home`
- The percentage of missing data was extremely low across all affected features.

### What This Means

Because the missing percentages were very small, removing rows would unnecessarily reduce the dataset size. Instead, imputation was later used to preserve data while maintaining dataset integrity.

## Exploring Unique Values and Categories

The next step involves examining the unique values present within each feature.

### Purpose of This Step

This analysis helps:

- Understand category distributions
- Detect inconsistent labels
- Identify binary, ordinal, and nominal variables
- Determine appropriate encoding strategies

Different types of categorical variables require different preprocessing techniques before modeling.

In [11]:
# Unique values + counts
for col in df.columns:
    print(f"\n🔹 Column: {col}")
    print(f"Unique values ({df[col].nunique()}):")
    print(df[col].unique())


🔹 Column: Hours_Studied
Unique values (41):
[23 19 24 29 25 17 21  9 10 14 22 15 12 20 11 13 16 18 31  8 26 28  4 35
 27 33 36 43 34  1 30  7 32  6 38  5  3  2 39 37 44]

🔹 Column: Attendance
Unique values (41):
[ 84  64  98  89  92  88  78  94  80  97  83  82  68  60  70  75  99  74
  65  62  91  90  66  69  72  63  61  86  77  71  67  87  73  96 100  81
  95  79  85  76  93]

🔹 Column: Parental_Involvement
Unique values (3):
<ArrowStringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str

🔹 Column: Access_to_Resources
Unique values (3):
<ArrowStringArray>
['High', 'Medium', 'Low']
Length: 3, dtype: str

🔹 Column: Extracurricular_Activities
Unique values (2):
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

🔹 Column: Sleep_Hours
Unique values (7):
[ 7  8  6 10  9  5  4]

🔹 Column: Previous_Scores
Unique values (51):
[ 73  59  91  98  65  89  68  50  80  71  88  87  97  72  74  70  82  58
  99  84 100  75  54  90  94  51  57  66  96  93  56  52  63  79  81  69
  95  60  92  

## Interpretation of Feature Categories

The unique value analysis revealed several different feature types within the dataset.

### Key Findings

The dataset contained:

- Continuous numerical variables
  - Example: `Hours_Studied`, `Attendance`
- Discrete numerical variables
  - Example: `Tutoring_Sessions`
- Binary categorical variables
  - Example: `Internet_Access`, `Learning_Disabilities`
- Ordinal categorical variables with ranked relationships
  - Example: `Low`, `Medium`, `High`
- Nominal categorical variables without ranking
  - Example: `Gender`, `School_Type`

### What This Means

Since different feature types contain different mathematical meanings, custom encoding strategies were required to preserve important relationships between categories while preparing the dataset for machine learning models.

## Feature Distribution Analysis

The distribution of each feature was analyzed using normalized value counts.

### Purpose of This Step

This analysis helps:

- Understand class balance
- Identify skewed variables
- Detect dominant categories
- Examine how student characteristics are distributed across the dataset

Understanding feature distributions improves preprocessing decisions and helps interpret later modeling results.

In [13]:
for col in df.columns:
    print(f"\n🔹 Distribution for: {col}")
    print(df[col].value_counts(normalize=True) * 100)


🔹 Distribution for: Hours_Studied
Hours_Studied
20    7.037990
19    6.674739
21    6.523384
23    6.220675
22    6.084456
18    6.069320
17    5.766611
24    5.403360
16    5.312547
15    4.767671
25    4.374149
14    4.071439
26    3.980627
27    3.466021
13    3.299531
12    2.906009
28    2.588164
11    2.209778
29    2.028152
30    1.861662
10    1.422733
9     1.301650
31    1.165431
8     0.877857
32    0.817315
7     0.771909
33    0.605418
34    0.438928
5     0.317845
35    0.302709
4     0.257303
6     0.257303
3     0.181626
36    0.166490
38    0.105948
39    0.105948
2     0.090813
37    0.090813
1     0.045406
43    0.015135
44    0.015135
Name: proportion, dtype: float64

🔹 Distribution for: Attendance
Attendance
67     2.875738
98     2.830331
76     2.800061
77     2.784925
64     2.754654
94     2.724383
84     2.648706
91     2.648706
79     2.648706
82     2.618435
68     2.573029
69     2.573029
80     2.557893
73     2.542758
96     2.542758
81     2.542758
72  

## Interpretation of Feature Distributions

The distribution analysis provided insight into how student characteristics and behaviors were represented throughout the dataset.

### Key Findings

- Most students fell within moderate study and attendance ranges.
- Categories such as `Medium` parental involvement and `Medium` access to resources were most common.
- Internet access was highly prevalent among students.
- Extreme exam scores were relatively rare compared to middle-range scores.

### What This Means

The dataset appears relatively balanced overall, though some high-performing exam score cases are less common. This later helps explain why the model performs best within the common score ranges and slightly underpredicts rare extreme values.

## Organizing Features by Data Type

The features were grouped into logical categories based on their data structure and meaning.

### Feature Categories

The dataset was separated into:

- Continuous numerical variables
- Discrete numerical variables
- Binary categorical variables
- Ordinal categorical variables
- Nominal categorical variables

### Why This Step Matters

Organizing variables by feature type allows preprocessing methods to be applied consistently and appropriately across the dataset.

In [15]:
continuous_numeric = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Physical_Activity"
    #"Exam_Score"
]

discrete_numeric = [
    "Tutoring_Sessions"
]

binary_numeric = [
    "Extracurricular_Activities",
    "Internet_Access",
    "Learning_Disabilities"
]

ordinal_categorical = [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality",
    "Parental_Education_Level",
    "Distance_from_Home",
    "Peer_Influence"
]

nominal_categorical = [
    "School_Type",
    "Gender"
]

## Encoding Categorical Variables

Machine learning models require numerical inputs, so categorical variables were converted into numerical representations using mapping techniques.

### Encoding Strategy

Different encoding methods were used depending on the feature type:

- Binary encoding for yes/no variables
- Ordinal encoding for ranked categories
- Custom mappings for educational levels and peer influence
- Numerical mapping for nominal variables

This approach preserves meaningful relationships between categories while preparing the dataset for modeling.

In [17]:
low_med_high_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

education_map = {
    "High School": 1,
    "College": 2,
    "Postgraduate": 3
}

distance_map = {
    "Near": 1,
    "Moderate": 2,
    "Far": 3
}

peer_map = {
    "Negative": -1,
    "Neutral": 0,
    "Positive": 1
}

binary_map = {
    "Yes": 1, 
    "No": 0
}

gender_map = {
    "Male": 1,
    "Female": 0
}

school_type_map = {
    "Public": 0,
    "Private": 1
}

In [19]:
for col in binary_numeric:
    df[col] = df[col].map(binary_map)
    
for col in [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality"
]:
    df[col] = df[col].map(low_med_high_map)

df["Parental_Education_Level"] = df["Parental_Education_Level"].map(education_map)
df["Distance_from_Home"] = df["Distance_from_Home"].map(distance_map)
df["Peer_Influence"] = df["Peer_Influence"].map(peer_map)
df["Gender"] = df["Gender"].map(gender_map)
df["School_Type"] = df["School_Type"].map(school_type_map)

In [21]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,1,3,0,7,73,1,1,0,1,2.0,0,1,3,0,1.0,1.0,1,67
1,19,64,1,2,0,8,59,1,1,2,2,2.0,0,-1,4,0,2.0,2.0,0,61
2,24,98,2,2,1,7,91,2,1,2,2,2.0,0,0,4,0,3.0,1.0,1,74
3,29,89,1,2,1,8,98,2,1,1,2,2.0,0,-1,4,0,1.0,2.0,1,71
4,19,92,2,2,1,6,65,2,1,3,2,3.0,0,0,4,0,2.0,1.0,0,70


## Interpretation of Encoded Features

After encoding, all categorical variables were successfully transformed into numerical representations.

### Key Results

- Ordinal relationships were preserved numerically.
- Binary variables were converted into 0/1 representations.
- Educational and behavioral categories were transformed into machine-readable formats.
- The dataset became fully compatible with machine learning algorithms.

### What This Means

The encoded dataset now contains consistent numerical features that can be interpreted mathematically by regression and ensemble learning models.

## Handling Missing Values with Imputation

To preserve dataset size and avoid losing valuable information, missing values were handled using imputation rather than row deletion.

### Imputation Method

The `SimpleImputer` with the `most_frequent` strategy was applied to ordinal categorical columns.

### Why Most Frequent Imputation?

This method was selected because:

- Missing percentages were extremely low
- The affected features were categorical/ordinal
- Replacing missing values with the most common category preserves distribution consistency

In [23]:
from sklearn.impute import SimpleImputer

ordinal_missing_cols = [
    "Parental_Education_Level",
    "Teacher_Quality",
    "Distance_from_Home"
]

imputer = SimpleImputer(strategy="most_frequent")

df[ordinal_missing_cols] = imputer.fit_transform(df[ordinal_missing_cols])

In [25]:
df[ordinal_missing_cols].isnull().sum()

Parental_Education_Level    0
Teacher_Quality             0
Distance_from_Home          0
dtype: int64

## Interpretation of Imputation Results

After imputation, all missing values were successfully removed from the dataset.

### Key Findings

- No remaining null values were present in the cleaned dataset.
- Dataset size was preserved without dropping observations.
- Category distributions remained stable after imputation.

### What This Means

The dataset is now complete, consistent, and ready for feature engineering and machine learning model training.

In [27]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,1,3,0,7,73,1,1,0,1,2.0,0,1,3,0,1.0,1.0,1,67
1,19,64,1,2,0,8,59,1,1,2,2,2.0,0,-1,4,0,2.0,2.0,0,61
2,24,98,2,2,1,7,91,2,1,2,2,2.0,0,0,4,0,3.0,1.0,1,74
3,29,89,1,2,1,8,98,2,1,1,2,2.0,0,-1,4,0,1.0,2.0,1,71
4,19,92,2,2,1,6,65,2,1,3,2,3.0,0,0,4,0,2.0,1.0,0,70


## Feature Engineering

A new feature called `Study_Efficiency` was engineered to capture the relationship between study time and sleep duration.

### Feature Formula

Study Efficiency was calculated as:

\[
\text{Study Efficiency} = \frac{\text{Hours Studied}}{\text{Sleep Hours} + 1}
\]

### Purpose of This Feature

This engineered variable was designed to:

- Capture productivity rather than raw study time
- Represent how efficiently students use their study hours
- Improve predictive power for exam score modeling

Feature engineering helps machine learning models identify deeper relationships within the data.

In [34]:
df["Study_Efficiency"] = df["Hours_Studied"] / (df["Sleep_Hours"] + 1)

In [36]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,...,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score,Study_Efficiency
0,23,84,1,3,0,7,73,1,1,0,...,2.0,0,1,3,0,1.0,1.0,1,67,2.875000
1,19,64,1,2,0,8,59,1,1,2,...,2.0,0,-1,4,0,2.0,2.0,0,61,2.111111
2,24,98,2,2,1,7,91,2,1,2,...,2.0,0,0,4,0,3.0,1.0,1,74,3.000000
3,29,89,1,2,1,8,98,2,1,1,...,2.0,0,-1,4,0,1.0,2.0,1,71,3.222222
4,19,92,2,2,1,6,65,2,1,3,...,3.0,0,0,4,0,2.0,1.0,0,70,2.714286


In [38]:
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percent
})

missing_df.sort_values(by="Missing %", ascending=False)

,Missing Count,Missing %
Hours_Studied,0,0.0
Teacher_Quality,0,0.0
Exam_Score,0,0.0
Gender,0,0.0
Distance_from_Home,0,0.0
Parental_Education_Level,0,0.0
Learning_Disabilities,0,0.0
Physical_Activity,0,0.0
Peer_Influence,0,0.0
School_Type,0,0.0


## Final Dataset Validation

A final validation step was performed to confirm dataset quality before exporting the cleaned dataset.

### Final Checks

- All missing values were removed
- Categorical variables were fully encoded
- Feature engineering was completed successfully
- The dataset contained only machine-learning-ready numerical values

### Outcome

The cleaned dataset was successfully prepared for:

- Exploratory Data Analysis (EDA)
- Regression modeling
- Ensemble learning
- Hyperparameter tuning
- Model evaluation and visualization

## Exporting the Cleaned Dataset

After preprocessing, encoding, imputation, and feature engineering, the cleaned dataset was exported as a new CSV file.

### Purpose

Saving the cleaned dataset separately allows:

- Reproducibility
- Faster modeling workflows
- Separation between raw and processed data
- Easier experimentation during model development

The exported dataset is used in the subsequent modeling and evaluation notebook.

In [40]:
df.to_csv("CleanedStudentPerformance.csv", index=False)

# Data Cleaning and Preprocessing Summary

In this notebook, the raw Student Performance Factors dataset was successfully transformed into a clean and machine-learning-ready dataset.

## Major Steps Completed

### Data Inspection
- Loaded and explored the raw dataset
- Analyzed dataset structure, dimensions, and data types

### Missing Value Analysis
- Identified incomplete features
- Measured missing value percentages
- Applied imputation to preserve dataset size

### Feature Exploration
- Examined unique values and category distributions
- Identified numerical, binary, ordinal, and nominal variables

### Encoding
- Converted categorical variables into numerical form
- Preserved ordinal relationships using custom mappings

### Feature Engineering
- Created the `Study_Efficiency` feature to capture productivity-related behavior

### Final Validation
- Verified that all missing values were removed
- Confirmed dataset consistency and modeling readiness

## Final Outcome

The final cleaned dataset is fully prepared for exploratory data analysis and machine learning modeling. The preprocessing pipeline improved data consistency, interpretability, and compatibility with ensemble regression models used later in the project.